# Regularización en Redes Neuronales

## Objetivo

Comprender los diferentes métodos de **regularización** aplicados a redes neuronales, incluyendo **L1**, **L2** y **Dropout**, para prevenir el sobreajuste (*overfitting*) y mejorar la capacidad de generalización de los modelos entrenados.

---

## 1. ¿Qué es la regularización?

En aprendizaje automático, **regularización** es una técnica utilizada para **evitar el sobreajuste** (*overfitting*) del modelo. El sobreajuste ocurre cuando la red neuronal aprende demasiado bien los patrones del conjunto de entrenamiento, incluyendo el ruido, y por lo tanto **no generaliza bien** a nuevos datos.

**Objetivo de la regularización:**
- Reducir la complejidad del modelo.
- Evitar que los pesos (parámetros) crezcan demasiado.
- Mejorar el rendimiento de validación y prueba.

La idea central es **añadir una penalización** a la función de pérdida (loss function) que dependa del tamaño de los pesos. De esta forma, la red prefiere soluciones más simples.

---

## 2. Tipos de Regularización

### Regularización L1 (Lasso)
- Agrega una penalización proporcional al **valor absoluto** de los pesos.
- Fomenta que muchos pesos se vuelvan exactamente **cero**, generando modelos más simples y **dispersos**.

$$L_{total} = L_{original} + \lambda \sum |w_i|$$

**Ventajas:** induce *sparsity* (pesos en cero).

**Ejemplo en Keras:**
```python
from tensorflow.keras import regularizers
Dense(64, activation='relu', kernel_regularizer=regularizers.l1(0.001))
```

---

### Regularización L2 (Ridge)
- Agrega una penalización proporcional al **cuadrado** de los pesos.
- Favorece pesos pequeños pero no necesariamente cero.

$$L_{total} = L_{original} + \lambda \sum w_i^2$$

**Ventajas:** estabilidad numérica, evita grandes oscilaciones en los pesos.

**Ejemplo en Keras:**
```python
Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))
```

---

### Regularización combinada L1 + L2 (Elastic Net)
- Combina los beneficios de L1 y L2.

```python
Dense(64, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.001))
```

---

### Dropout
- Técnica distinta a L1/L2.
- Durante el entrenamiento, **“apaga” aleatoriamente** un porcentaje de neuronas para evitar que la red dependa demasiado de rutas específicas.

Por ejemplo, un `Dropout(0.3)` apagará el 30% de las neuronas en cada iteración.

```python
from tensorflow.keras.layers import Dropout
model.add(Dropout(0.3))
```

**Ventajas:** mejora la generalización del modelo.

**Desventaja:** puede ralentizar el entrenamiento.

---

In [ ]:
import numpy as np
# Ejemplo en Google Colab
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, OneHotEncoder

iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=iris.target)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=np.argmax(y_train, axis=1))
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Modelo regularizado
model = models.Sequential([
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001), input_shape=(X.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(y.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val), verbose=0)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Exactitud en prueba: {test_acc:.4f}')

## Conclusión
La **regularización** permite mejorar la capacidad de generalización de las redes neuronales.

- **L1:** fuerza la simplicidad del modelo reduciendo algunos pesos a cero.
- **L2:** penaliza pesos grandes, favoreciendo estabilidad numérica.
- **Dropout:** apaga neuronas aleatoriamente, reduciendo el sobreajuste.

*Pruebe diferentes valores de `lambda` y tasas de `Dropout` para observar su efecto en la precisión.*